# **Titanic Survival Prediction using Random Forest Classifier**
- The sinking of the Titanic is one of the most infamous shipwrecks in history.
- On April 15, 1912, during her maiden voyage, the widely considered ?unsinkable? RMS Titanic sank after colliding with an iceberg. Unfortunately, there weren?t enough lifeboats for everyone onboard, resulting in the death of 1502 out of 2224 passengers and crew.
- While there was some element of luck involved in surviving, it seems some groups of people were more likely to survive than others.

In this challenge, we ask you to build a predictive model that answers the question :
```bash
What sorts of people were more likely to survive?
```
using passenger data (ie name, age, gender, socio-economic class, etc).

## 1. Importing dependancies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Importing and preprocessing data
The data is divided into 3 parts,
1. ```train.csv``` - This data is the training data for the project. It ill furthr be divided into two parts, training and testing for building the model.
2. ```test.csv``` - This data will be used for the actual testing of the model on the final test dataset.

In [ ]:
titanic_data = pd.read_csv('train.csv')

In [ ]:
titanic_data

From the above command, we can derive various charactristics of the people who have survived the titanic ship wreck.
Let me discuss the features that the dataset is depicting :
1. **Passenger ID**
2. **Survived**
  - ```0 (not survived)```
  - ```1 (survived)```
3. **Pclass** - The class of the passenger
  - ```1 (high class)```
  - ```2 (middle class)```
  - ```3 (lower class)```
4. **Name**
5. **Sex**
6. **Age**
7. **SibSp** - The no. of siblings/spouse of the passenger.
  - ```Sibling``` : brother, sister, stepbrother, stepsister.
  - ```Spouse``` : husband, wife (mistresses and fianc?s were ignored).
8. **Parch** - This defines the no.of parent/child of the passengers
  - ```Parent``` : Mother, Father.
  - ```Child``` : daughter, son, stepdaughter, stepson.
  - ```Note``` : Some children travelled with thir nannies hence theyre parch = 0.
9.  **Ticket** - The ticket number.
10. **Fare** - The price of the ticket.
11. **Cabin** - The cabin number.
12. **Embarked** - Port of embarkation.
  - ```C``` : Cherbourg
  - ```Q``` : Queenstown
  - ```S``` : Southampton

In [ ]:
titanic_data.info()

here, some things important to understand are :
1. The age values present are less, if you look above there are several NaN values. I am going to deal with that using the estimators which i've done for stratified data.
2. The cabon number is also not present for a lot of passengers, approximately for 650 passengers.
3. The place from which 2 passengers have boarded is yet unknown.

In [ ]:
titanic_data.describe()

In [ ]:
titanic_corr = titanic_data.drop(columns=['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'])
sns.heatmap(titanic_corr.corr(), cmap="YlGnBu")
plt.show()

From this heatmap we can find the correlation of the features given in the titanic dataset
Here are my conclusions
1. The survival rate is highly correlated to the passenger class and the fare price.
2. Obviously, the passenger class and the fare price are also highly correlated.
3. The age and the survival rate are also related.
4. The sibling/parent or wife/spouse of the passenger seem to have a less survival rate than the passenger.

## 3. Stratified Shuffle Split

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2)
for train_indices, test_indices in split.split(titanic_data, titanic_data[['Survived', 'Pclass', 'Sex']]):
  strat_train_set = titanic_data.loc[train_indices]
  strat_test_set = titanic_data.loc[test_indices]


In [ ]:
strat_train_set

In [ ]:
plt.subplot(1, 2, 1)
strat_train_set['Survived'].hist()
strat_train_set['Pclass'].hist()

plt.subplot(1, 2, 2)
strat_test_set['Survived'].hist()
strat_test_set['Pclass'].hist()

plt.show()

In [ ]:
strat_train_set.info()

## 4. Handling Missing Values

- ```BaseEstimator``` - This is the base class for all estimators in scikit-learn that provide hyper-parameter tuning like *get_params* and *set_params*.
- ```TranformMixin``` - This is the base class to create transofmers in scikit-learn.
- ```class AgeImputer()``` - This is the base custom class designed to handle missing values of parameter *Age*.
- ```def fit()``` - The fit method is used by the transformer to learn from the data, in this case the transformer does not need to learn anything hence I used return self.
- ```def transform``` - This method is used to fit the missing data of age into columns by replacing the NaN values by the mean of the data

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer

class AgeImputer(BaseEstimator, TransformerMixin):

  def fit(self, X, Y=None):
    return self

  def transform(self, X):
    imputer = SimpleImputer(strategy='mean')
    X['Age'] = imputer.fit_transform(X[['Age']])
    return X


In [ ]:
from sklearn.preprocessing import OneHotEncoder

class FeatureEncoder(BaseEstimator, TransformerMixin):

  def fit(self, X, Y=None):
    return self

  def transform(self, X):
    encoder = OneHotEncoder()
    matrix = encoder.fit_transform(X[['Embarked']]).toarray()

    column_names = ['C', 'S', 'Q', 'N']

    for i in range(len(matrix.T)):
      X[column_names[i]] = matrix.T[i]

    matrix = encoder.fit_transform(X[['Sex']]).toarray()

    column_names = ['Female', 'Male']

    for i in range(len(matrix.T)):
      X[column_names[i]] = matrix.T[i]

    return X

In [ ]:
class FeatureDropper(BaseEstimator, TransformerMixin):

  def fit(self, X, Y=None):
    return self

  def transform(self, X):
    return X.drop(columns=['Embarked', 'Name', 'Ticket', 'Cabin', 'Sex', 'N'], axis=1, errors='ignore')

## 5. Creating pipeline

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("ageimputer", AgeImputer()),
    ("featureencoder", FeatureEncoder()),
    ("featuredropper", FeatureDropper())
])

In [ ]:
strat_train_set = pipeline.fit_transform(strat_train_set)
strat_train_set

In [ ]:
strat_train_set.info()

## 6. Standardizing the pipelined data

In [ ]:
from sklearn.preprocessing import StandardScaler

X = strat_train_set.drop(['Survived'], axis=1)
y = strat_train_set['Survived']

scaler = StandardScaler()
X_data = scaler.fit_transform(X)
y_data = y.to_numpy()

## 7. Implementing Random Forest Classifier and GridSearch

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

clf = RandomForestClassifier()

param_grid = [
    {"n_estimators": [10, 100, 200, 500], "max_depth": [None, 5, 10], "min_samples_split": [2, 3, 4]}
]

grid_search = GridSearchCV(clf, param_grid, cv=5, scoring="accuracy", return_train_score=True)
grid_search.fit(X_data, y_data)


In [ ]:
final_clf = grid_search.best_estimator_

In [ ]:
final_clf

In [ ]:
strat_test_set = pipeline.fit_transform(strat_test_set)

In [ ]:
X_test = strat_test_set.drop(['Survived'], axis=1)
y_test = strat_test_set['Survived']

scaler = StandardScaler()
X_data_test = scaler.fit_transform(X_test)
y_data_test = y_test.to_numpy()

In [ ]:
final_clf.score(X_data_test, y_data_test)

In [ ]:
final_data = pipeline.fit_transform(titanic_data)

In [ ]:
final_data

In [ ]:
X_final = final_data.drop(['Survived'], axis=1)
y_final = final_data['Survived']

scaler = StandardScaler()
X_data_final = scaler.fit_transform(X_final)
y_data_final = y_final.to_numpy()

In [ ]:
prod_clf = RandomForestClassifier()

param_grid = [
    {"n_estimators": [10, 100, 200, 500], "max_depth": [None, 5, 10], "min_samples_split": [2, 3, 4]}
]

grid_search = GridSearchCV(prod_clf, param_grid, cv=5, scoring="accuracy", return_train_score=True)
grid_search.fit(X_data_final, y_data_final)

## 8. Prediction

In [ ]:
final_clf.score(X_data_final, y_data_final)

In [ ]:
prod_final_clf = grid_search.best_estimator_

In [ ]:
titanic_test_data = pd.read_csv('/content/test.csv')

In [ ]:
final_test_data = pipeline.fit_transform(titanic_test_data)

In [ ]:
final_test_data

In [ ]:
final_test_data.info()

In [ ]:
X_final_test = final_test_data
X_final_test = X_final_test.fillna(method='ffill')

scaler = StandardScaler()
X_data_final_test = scaler.fit_transform(X_final_test)

In [ ]:
predictions = final_clf.predict(X_data_final_test)

In [ ]:
final_df = pd.DataFrame(titanic_test_data['PassengerId'])
final_df['Survived'] = predictions
final_df.to_csv('/content/predcitions.csv')

In [ ]:
final_df

In [ ]:
len(final_df['PassengerId'])

In [ ]:
print("Titanic Wreck Survival Prediction")
option = 0
ID = input("Enter the Passenger ID to know if he/she survived : ")
ID = int(ID)
if ID in final_df['PassengerId'].values:
  passenger_row = final_df.loc[final_df['PassengerId']==ID]
  survived = passenger_row['Survived'].values[0]

  if survived == 1:
    print(f"Passenger ID {ID} survived: {'Yes' if survived == 1 else 'No'}")
else:
    print("Invalid ID")

---

If you have any doubts related to this project you can email me on ```venomc381@gmail.com```